# 01 Data Generation

This notebook designs a retail customer-service intent and slot schema, then creates 400 synthetic JSON samples using weighted business distributions. It includes deterministic seed handling and explicit edge-case injection.

In [1]:
from __future__ import annotations
import json
import logging
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')
logger = logging.getLogger('data_generation')
ROOT = Path.cwd()
SEED = 20260605
N_SAMPLES = 400
EDGE_CASE_RATE = 0.30

## Intent and Slot Schema

The weights below approximate a realistic retail support queue: order tracking, delivery delays, cancellations, refunds, returns, and payments are more frequent than invoice or subscription requests.

In [2]:
@dataclass(frozen=True)
class SlotDef:
    name: str
    dtype: str
    required: bool
    description: str
    examples: List[str]

@dataclass(frozen=True)
class IntentDef:
    name: str
    weight: float
    description: str
    slots: List[SlotDef]

def slot(name: str, dtype: str, required: bool, description: str, examples: Sequence[str]) -> SlotDef:
    return SlotDef(name, dtype, required, description, list(examples))

INTENTS = [
    IntentDef('order_status', 0.155, 'Customer asks for current order state.', [slot('order_id','string',True,'Retail order identifier.',['ORD382910']), slot('phone_last4','string',False,'Last four account phone digits.',['4421']), slot('channel','string',False,'Purchase channel.',['app'])]),
    IntentDef('delivery_delay', 0.120, 'Customer reports delayed delivery.', [slot('order_id','string',True,'Retail order identifier.',['ORD700423']), slot('expected_delivery_date','date',False,'Promised delivery date.',['2026-06-07']), slot('delivery_slot','string',False,'Selected time slot.',['8 AM - 10 AM']), slot('city','string',False,'Delivery city.',['Bengaluru'])]),
    IntentDef('cancel_order', 0.100, 'Customer wants to cancel order or items.', [slot('order_id','string',True,'Retail order identifier.',['ORD811204']), slot('cancellation_reason','string',False,'Cancellation reason.',['ordered by mistake']), slot('products','list[string]',False,'Products to cancel.',['milk'])]),
    IntentDef('refund_status', 0.090, 'Customer asks about refund progress.', [slot('refund_id','string',False,'Refund identifier.',['RFN48201']), slot('order_id','string',True,'Retail order identifier.',['ORD554812']), slot('payment_method','string',False,'Original payment method.',['UPI'])]),
    IntentDef('return_item', 0.080, 'Customer wants to return items.', [slot('order_id','string',True,'Retail order identifier.',['ORD602918']), slot('products','list[string]',True,'Products to return.',['detergent']), slot('return_reason','string',False,'Reason for return.',['damaged item']), slot('pickup_address','string',False,'Return pickup address.',['same as delivery address'])]),
    IntentDef('payment_issue', 0.075, 'Customer reports payment issue.', [slot('order_id','string',False,'Retail order identifier.',['ORD309855']), slot('transaction_id','string',False,'Payment transaction identifier.',['TXN88291120']), slot('payment_method','string',True,'Payment method used.',['UPI']), slot('issue_type','string',True,'Type of payment issue.',['duplicate_charge'])]),
    IntentDef('modify_order', 0.070, 'Customer wants to change order before dispatch.', [slot('order_id','string',True,'Retail order identifier.',['ORD915771']), slot('products','list[string]',False,'Items to change.',['bread']), slot('quantity_change','string',False,'Requested quantity change.',['add 1 unit']), slot('delivery_slot','string',False,'Requested delivery slot.',['6 PM - 8 PM'])]),
    IntentDef('product_availability', 0.070, 'Customer checks product availability.', [slot('products','list[string]',True,'Products being searched.',['tomatoes']), slot('pincode','string',False,'Delivery pincode.',['560001']), slot('store_id','string',False,'Store identifier.',['STR1024'])]),
    IntentDef('complaint_registration', 0.060, 'Customer registers complaint.', [slot('order_id','string',False,'Retail order identifier.',['ORD150920']), slot('complaint_type','string',True,'Complaint category.',['missing_item']), slot('products','list[string]',False,'Products involved.',['paneer']), slot('description','string',False,'Issue description.',['one item was missing'])]),
    IntentDef('exchange_item', 0.050, 'Customer requests exchange.', [slot('order_id','string',True,'Retail order identifier.',['ORD445901']), slot('products','list[string]',True,'Products for exchange.',['shirt']), slot('exchange_reason','string',False,'Exchange reason.',['wrong size']), slot('preferred_variant','string',False,'Replacement variant.',['size L'])]),
    IntentDef('store_locator', 0.040, 'Customer asks for nearest store.', [slot('city','string',False,'Customer city.',['Pune']), slot('pincode','string',False,'Lookup pincode.',['411014']), slot('store_type','string',False,'Type of store.',['pickup_point'])]),
    IntentDef('delivery_address_update', 0.030, 'Customer updates delivery address.', [slot('order_id','string',True,'Retail order identifier.',['ORD228145']), slot('new_address','string',True,'Updated delivery address.',['Flat 402']), slot('city','string',False,'Delivery city.',['Chennai']), slot('pincode','string',False,'Delivery pincode.',['600042'])]),
    IntentDef('coupon_issue', 0.025, 'Customer reports coupon issue.', [slot('coupon_code','string',True,'Coupon code.',['SAVE100']), slot('order_id','string',False,'Retail order identifier.',['ORD456100']), slot('cart_value','number',False,'Cart value.',['1499'])]),
    IntentDef('subscription_issue', 0.020, 'Customer asks about subscription benefits.', [slot('subscription_id','string',False,'Subscription identifier.',['SUB88201']), slot('issue_type','string',True,'Subscription issue type.',['benefit_not_applied']), slot('phone_last4','string',False,'Last four phone digits.',['1212'])]),
    IntentDef('invoice_request', 0.015, 'Customer asks for invoice.', [slot('order_id','string',True,'Retail order identifier.',['ORD901882']), slot('invoice_type','string',False,'Invoice type.',['GST']), slot('email','string',False,'Invoice email.',['customer@example.com'])]),
]

assert abs(sum(intent.weight for intent in INTENTS) - 1.0) < 1e-9
schema = {'schema_version': '1.0', 'domain': 'retail_delivery_customer_service', 'seed': SEED, 'intents': [asdict(intent) for intent in INTENTS]}
(ROOT / 'schema.json').write_text(json.dumps(schema, indent=2), encoding='utf-8')
logger.info('Saved schema with %d intents', len(INTENTS))

INFO:Saved schema with 15 intents


## Data Generator

Reusable functions generate base slots, inject edge cases, render utterances and target responses, then perform deterministic splitting.

In [3]:
PRODUCTS = ['milk','curd','paneer','eggs','bread','basmati rice','atta','toor dal','cooking oil','apples','bananas','tomatoes','onions','potatoes','baby diapers','detergent','shampoo','toothpaste','pressure cooker','t-shirt','almonds','green tea']
PAYMENT_METHODS = ['UPI','credit_card','debit_card','net_banking','wallet','cash_on_delivery']
CITIES = ['Bengaluru','Mumbai','Delhi','Pune','Hyderabad','Chennai','Kolkata','Ahmedabad','Gurugram']
CHANNELS = ['app','website','store','call_center']
DELIVERY_SLOTS = ['6 AM - 8 AM','8 AM - 10 AM','10 AM - 12 PM','2 PM - 4 PM','6 PM - 8 PM','8 PM - 10 PM']
EDGE_CASE_TYPES = ['missing_required_slot','null_value','empty_value','invalid_id','multiple_products','partial_information','duplicate_entities','ambiguous_request']

def choose(rng: random.Random, values: Sequence[Any]) -> Any:
    return values[rng.randrange(len(values))]

def order_id(rng: random.Random) -> str: return f'ORD{rng.randrange(100000, 999999)}'
def phone_last4(rng: random.Random) -> str: return f'{rng.randrange(0, 10000):04d}'
def pincode(rng: random.Random) -> str: return f"{choose(rng, ['560','400','110','411','500','600','700','380','122'])}{rng.randrange(100, 999)}"
def products(rng: random.Random, min_items: int = 1, max_items: int = 3) -> List[str]: return rng.sample(PRODUCTS, rng.randrange(min_items, max_items + 1))
def required_slots(intent_name: str) -> List[str]:
    return [s.name for intent in INTENTS if intent.name == intent_name for s in intent.slots if s.required]

def generate_base_slots(intent_name: str, rng: random.Random) -> Dict[str, Any]:
    oid = order_id(rng)
    mapping = {
        'order_status': {'order_id': oid, 'phone_last4': phone_last4(rng), 'channel': choose(rng, CHANNELS)},
        'delivery_delay': {'order_id': oid, 'expected_delivery_date': '2026-06-%02d' % rng.randrange(6, 15), 'delivery_slot': choose(rng, DELIVERY_SLOTS), 'city': choose(rng, CITIES)},
        'cancel_order': {'order_id': oid, 'cancellation_reason': choose(rng, ['ordered by mistake','delivery too late','duplicate order']), 'products': products(rng, 1, 2)},
        'refund_status': {'refund_id': f'RFN{rng.randrange(10000,99999)}', 'order_id': oid, 'payment_method': choose(rng, PAYMENT_METHODS)},
        'return_item': {'order_id': oid, 'products': products(rng, 1, 3), 'return_reason': choose(rng, ['damaged item','expired product','wrong item delivered']), 'pickup_address': 'same as delivery address'},
        'payment_issue': {'order_id': oid, 'transaction_id': f'TXN{rng.randrange(10000000,99999999)}', 'payment_method': choose(rng, PAYMENT_METHODS), 'issue_type': choose(rng, ['charged_but_order_not_placed','duplicate_charge','payment_failed'])},
        'modify_order': {'order_id': oid, 'products': products(rng, 1, 2), 'quantity_change': choose(rng, ['increase to 2 packs','remove 1 unit','add 1 unit']), 'delivery_slot': choose(rng, DELIVERY_SLOTS)},
        'product_availability': {'products': products(rng, 1, 3), 'pincode': pincode(rng), 'store_id': f'STR{rng.randrange(1000,9999)}'},
        'complaint_registration': {'order_id': oid, 'complaint_type': choose(rng, ['missing_item','poor_quality','wrong_item','late_delivery']), 'products': products(rng, 1, 2), 'description': choose(rng, ['one item was missing','package was leaking','product was stale'])},
        'exchange_item': {'order_id': oid, 'products': products(rng, 1, 2), 'exchange_reason': choose(rng, ['wrong size','wrong color','defective item']), 'preferred_variant': choose(rng, ['size L','red color','1 kg pack'])},
        'store_locator': {'city': choose(rng, CITIES), 'pincode': pincode(rng), 'store_type': choose(rng, ['supermarket','express_store','pickup_point'])},
        'delivery_address_update': {'order_id': oid, 'new_address': choose(rng, ['Flat 402, Lake View Apt','Tower B, Prestige Park','Office reception desk']), 'city': choose(rng, CITIES), 'pincode': pincode(rng)},
        'coupon_issue': {'coupon_code': choose(rng, ['SAVE100','FRESH20','FREESHIP','WEEKEND200']), 'order_id': oid, 'cart_value': rng.randrange(299, 3999)},
        'subscription_issue': {'subscription_id': f'SUB{rng.randrange(10000,99999)}', 'issue_type': choose(rng, ['benefit_not_applied','renewal_failed','membership_not_visible']), 'phone_last4': phone_last4(rng)},
        'invoice_request': {'order_id': oid, 'invoice_type': choose(rng, ['standard','GST','duplicate_copy']), 'email': f'customer{rng.randrange(100,999)}@example.com'},
    }
    return mapping[intent_name]

def inject_edge_case(intent_name: str, slots: Dict[str, Any], rng: random.Random) -> Tuple[Dict[str, Any], List[str]]:
    updated, cases = dict(slots), rng.sample(EDGE_CASE_TYPES, rng.randrange(1, 3))
    for case in cases:
        keys = list(updated)
        if case == 'missing_required_slot':
            candidates = [name for name in required_slots(intent_name) if name in updated]
            if candidates: updated.pop(choose(rng, candidates), None)
        elif case == 'null_value' and keys: updated[choose(rng, keys)] = None
        elif case == 'empty_value': updated[choose(rng, keys) if keys else 'notes'] = ''
        elif case == 'invalid_id': updated[choose(rng, [k for k in keys if k.endswith('_id')] or ['order_id'])] = choose(rng, ['12345','ORDER-??','ORD','INVALID_ID'])
        elif case == 'multiple_products': updated['products'] = products(rng, 2, 4)
        elif case == 'partial_information': updated = {k: updated[k] for k in keys[:max(1, len(keys)//2)]}
        elif case == 'duplicate_entities':
            vals = list(updated.get('products') or products(rng, 1, 2)); vals.append(vals[0]); updated['products'] = vals
        elif case == 'ambiguous_request': updated['ambiguous_reference'] = choose(rng, ['the last order','same item','yesterday delivery','the second one'])
    return updated, cases

def render_customer_utterance(intent: str, slots: Dict[str, Any], edge_cases: Sequence[str]) -> str:
    if 'ambiguous_request' in edge_cases: return f"Can you help with {slots.get('ambiguous_reference', 'that order')}?"
    return f"I need help with {intent.replace('_', ' ')} for {slots.get('order_id', slots.get('products', 'my account'))}."

def render_agent_response(intent: str, slots: Dict[str, Any]) -> str:
    oid = slots.get('order_id')
    return f"I can help with {intent.replace('_', ' ')}" + (f" for order {oid}." if oid else '. Please share the missing details for verification.')

In [4]:
def generate_samples(seed: int = SEED, count: int = N_SAMPLES, edge_case_rate: float = EDGE_CASE_RATE) -> List[Dict[str, Any]]:
    py_rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    names = [intent.name for intent in INTENTS]
    weights = np.array([intent.weight for intent in INTENTS], dtype=float)
    intent_names = np_rng.choice(names, size=count, replace=True, p=weights / weights.sum())
    edge_indices = set(np_rng.choice(np.arange(count), size=int(round(count * edge_case_rate)), replace=False))
    samples = []
    for i, intent_name in enumerate(intent_names, start=1):
        slots = generate_base_slots(str(intent_name), py_rng)
        edge_types: List[str] = []
        if i - 1 in edge_indices:
            slots, edge_types = inject_edge_case(str(intent_name), slots, py_rng)
        samples.append({'sample_id': f'SMP{i:04d}', 'intent': str(intent_name), 'slots': slots, 'customer_utterance': render_customer_utterance(str(intent_name), slots, edge_types), 'target_text': render_agent_response(str(intent_name), slots), 'edge_case': bool(edge_types), 'edge_case_types': edge_types, 'metadata': {'domain': 'retail_delivery_customer_service', 'schema_version': '1.0', 'seed': seed, 'source': 'synthetic_weighted_business_distribution'}})
    return samples

def split_samples(samples: Sequence[Dict[str, Any]], seed: int = SEED) -> Dict[str, List[Dict[str, Any]]]:
    rng = np.random.default_rng(seed)
    indices = np.arange(len(samples)); rng.shuffle(indices)
    bounds = {'train': indices[:280], 'validation': indices[280:340], 'test': indices[340:]}
    result = {}
    for split, idxs in bounds.items():
        result[split] = [dict(samples[int(i)], metadata={**samples[int(i)]['metadata'], 'split': split}) for i in idxs]
    return result

samples = generate_samples()
splits = split_samples(samples)
for split, records in splits.items():
    (ROOT / f'{split}.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
logger.info('Saved train=%d validation=%d test=%d', len(splits['train']), len(splits['validation']), len(splits['test']))

INFO:Saved train=280 validation=60 test=60


In [5]:
all_records = [record for records in splits.values() for record in records]
df = pd.DataFrame(all_records)
assert len(df) == 400
assert df['edge_case'].mean() >= 0.25
assert set(splits) == {'train', 'validation', 'test'}
display(df.groupby('intent').size().sort_values(ascending=False).to_frame('count'))
display(df.groupby('edge_case').size().to_frame('count'))

,count
intent,
order_status,49
delivery_delay,48
return_item,39
cancel_order,37
product_availability,37
modify_order,30
refund_status,30
complaint_registration,29
payment_issue,25


,count
edge_case,
False,280
True,120
